# Bile Acid Hydroxylation Model: Product-Level Predictions

Previous models predicted enzyme-amine activity by aggregating across all bile acid products (binary: any product active?). Analysis showed **20% of enzyme-amine pairs have mixed activity** across bile acid hydroxylation patterns. By predicting at the product level (enzyme x amine x bile acid), we gain:
- More training samples (~10k vs ~3.3k)
- A third feature dimension (bile acid hydroxylation) that captures real variation
- Finer-grained predictions

### Feature Architecture
| Block | Description | Dims |
|-------|-------------|------|
| Enzyme | Top 4 from previous analysis | 1024-2048 |
| Amine | physchem_onehot, MolT5-small, MolT5-base | 41, 512, 768 |
| Bile acid | Positional encoding (C3/C7/C12 status + counts) | 12 |

### Bile Acid Positional Encoding (12 dims)
- **Positions (9 dims):** C3, C7, C12 each encoded as [alpha-OH, keto, unspecified]
- **Counts (3 dims):** n_hydroxyl, n_ketone, is_specific
- Specific patterns (3a7k, etc.) get full positional info
- General categories (Mono/Di/Tri) get unspecified positions + count features only

### Experiment Design
- 4 enzyme repr × 3 amine repr × 3 models (XGBoost + RF + MLP) × 10 splits = **360 experiments**

## 1. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import h5py
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    precision_score, recall_score, accuracy_score, log_loss
)
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors

from Bio import SeqIO

DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../outputs")

BA_DIR = OUTPUT_DIR / "model_outputs" / "bile_acid_hydroxylation"
BA_DIR.mkdir(parents=True, exist_ok=True)

SEEDS = [42, 123, 456, 789, 1011, 2022, 3033, 4044, 5055, 6066]

print(f"Output directory: {BA_DIR}")

## 2. Load Data (Product-Level) & Build Bile Acid Encoding

In [ ]:
%%time
# --- Per-residue ProtT5 embeddings ---
h5_path = DATA_DIR / "Seqs_list_total_per_residue.h5"
per_residue_embeddings = {}
with h5py.File(h5_path, 'r') as f:
    for key in f.keys():
        uniprot_id = key.split('_')[-1]
        per_residue_embeddings[uniprot_id] = f[key][:]
print(f"Per-residue embeddings: {len(per_residue_embeddings)} enzymes")

# --- Full protein embeddings (mean-pooled) ---
h5_full = DATA_DIR / "Seqs_list_total.h5"
full_embeddings = {}
with h5py.File(h5_full, 'r') as f:
    for key in f.keys():
        uniprot_id = key.split('_')[-1]
        full_embeddings[uniprot_id] = f[key][:]
print(f"Full protein embeddings: {len(full_embeddings)} enzymes")

# --- Conservation scores ---
df_cons = pd.read_csv(OUTPUT_DIR / "conservation_scores.csv")
core_mask = df_cons['gap_fraction'] < 0.5
df_core = df_cons[core_mask].copy()
print(f"Conservation: {len(df_core)} core positions (gap < 0.5)")

# --- MSA alignment mapping ---
alignment_to_seq = {}
for record in SeqIO.parse(OUTPUT_DIR / "bsh_aligned.fasta", 'fasta'):
    parts = record.id.split('_')
    uniprot_id = parts[-1] if len(parts) > 1 else record.id
    seq_pos = 0
    mapping = {}
    for aln_pos, char in enumerate(str(record.seq)):
        if char != '-':
            mapping[aln_pos] = seq_pos
            seq_pos += 1
    alignment_to_seq[uniprot_id] = mapping
print(f"Alignment mappings: {len(alignment_to_seq)} sequences")

overlap = set(alignment_to_seq.keys()) & set(per_residue_embeddings.keys())
print(f"Enzymes with alignment + embeddings: {len(overlap)}")

# --- Activity labels (PRODUCT-LEVEL, no aggregation) ---
df_activity = pd.read_csv(OUTPUT_DIR / "enzyme_amine_activity.csv")
controls = ['CTRL1', 'CTRL2', 'CTRL3', 'CTRL4', 'CTRL5', 'CTRL6', 'CTRL7']
canonical = ['taurine', 'glycine']
df_activity = df_activity[~df_activity['Enzyme'].isin(controls)]
df_activity = df_activity[~df_activity['Amine'].isin(canonical)]
print(f"Product-level rows (after removing controls + canonical): {len(df_activity)}")
print(f"  Active: {df_activity['active_approach2'].sum()}, Inactive: {(~df_activity['active_approach2']).sum()} ({df_activity['active_approach2'].mean():.1%} active)")

# --- Amine SMILES ---
df_smiles = pd.read_excel(DATA_DIR / "bsh_reactants_SMILES_corrected.xlsx")

name_map = {
    '2,3-Diaminopropinoic Acid': '2,3_diaminopropionic acid',
    '2-aminophenol': '2_aminophenol',
    '3-methoxytyramine HCl': '3_methoxytyramine',
    '4-aminophenol': '4_aminophenol',
    'L-Alanine': 'alanine',
    'L-Arginine': 'arginine',
    'Asparagine': 'asparagine',
    'Cadaverine': 'cadaverine',
    'L-Citrulline': 'citrulline',
    'L-Cysteine': 'cysteine',
    'Dopamine HCl': 'dopamine',
    'gamma-Aminobutyric acid >99%': 'gaba',
    'L-Glutamine': 'glutamine',
    'Glycyl-L-Valine': 'glyglycine',
    'L-Histidine': 'histidine',
    'L-Lysine': 'lysine',
    'L-Methionine': 'methionine',
    'L-Ornithine monohydrochloride': 'ornithine',
    'L-Phenylalanine': 'phenylalanine',
    'DL-Proline': 'proline',
    'Putrescine': 'putrescine',
    'L-Serine': 'serine',
    'L-Threonine': 'threonine',
    'Tryptamine': 'tryptamine',
}

amine_mols = {}
for _, row in df_smiles.iterrows():
    name = row['Compound_Name']
    smiles = row['SMILES']
    norm_name = name_map.get(name, name.lower().replace(' ', '_').replace('-', '_'))
    if pd.isna(smiles):
        continue
    smiles_clean = smiles.split('.')[0]
    mol = Chem.MolFromSmiles(smiles_clean)
    if mol is not None:
        amine_mols[norm_name] = mol

amines_needed = df_activity['Amine'].unique()
print(f"Amines needed: {len(amines_needed)}, parsed: {len(amine_mols)}")
missing_amines = set(amines_needed) - set(amine_mols.keys())
print(f"Missing from SMILES: {missing_amines}")

In [ ]:
# --- Bile Acid Positional Encoding ---
# Merge 3a,7a,12k and 3a7a12k (same bile acid, different formatting)
df_activity['Hydroxyl'] = df_activity['Hydroxyl'].replace('3a,7a,12k', '3a7a12k')

print("Hydroxylation patterns after merge:")
print(df_activity['Hydroxyl'].value_counts().sort_index())

# Positional encoding: encode what's at C3, C7, C12
# Each position: [is_alpha_OH, is_keto, is_unspecified]
# Plus count features: n_hydroxyl, n_ketone, is_specific
#
# "a" = alpha-hydroxyl (counts as -OH), "k" = ketone (doesn't count as -OH)
# Classification by hydroxyl count only:
#   3a7k -> Mono (1 -OH at C3, 1 keto at C7)
#   3k7a -> Mono (1 keto at C3, 1 -OH at C7)
#   3a12k -> Mono (1 -OH at C3, 1 keto at C12)
#   3k12a -> Mono (1 keto at C3, 1 -OH at C12)
#   3a7a12k -> Di (2 -OH at C3+C7, 1 keto at C12)

# Define positional info for specific patterns
# Format: {pattern: (C3_status, C7_status, C12_status, n_OH, n_keto)}
# Status: 'a' = alpha-OH, 'k' = keto, None = unspecified
PATTERN_INFO = {
    '3a7k':    ('a', 'k', None, 1, 1),
    '3k7a':    ('k', 'a', None, 1, 1),
    '3a12k':   ('a', None, 'k', 1, 1),
    '3k12a':   ('k', None, 'a', 1, 1),
    '3a7a12k': ('a', 'a', 'k',  2, 1),
    'Mono':    (None, None, None, 1, 0),
    'Di':      (None, None, None, 2, 0),
    'Tri':     (None, None, None, 3, 0),
}

BA_FEATURE_NAMES = [
    'C3_aOH', 'C3_keto', 'C3_unspec',
    'C7_aOH', 'C7_keto', 'C7_unspec',
    'C12_aOH', 'C12_keto', 'C12_unspec',
    'n_OH', 'n_keto', 'is_specific',
]
BA_DIM = len(BA_FEATURE_NAMES)  # 12

def encode_bile_acid(hydroxyl_value):
    """Positional encoding: C3/C7/C12 status + counts = 12 dims."""
    vec = np.zeros(BA_DIM, dtype=np.float32)
    info = PATTERN_INFO.get(hydroxyl_value)
    if info is None:
        return vec
    
    c3, c7, c12, n_oh, n_keto = info
    
    # Encode each position: [alpha-OH, keto, unspecified]
    for pos_idx, status in enumerate([c3, c7, c12]):
        base = pos_idx * 3
        if status == 'a':
            vec[base] = 1.0      # alpha-OH
        elif status == 'k':
            vec[base + 1] = 1.0  # keto
        else:
            vec[base + 2] = 1.0  # unspecified
    
    # Count features
    vec[9] = n_oh       # n_hydroxyl (1-3)
    vec[10] = n_keto    # n_ketone (0-1)
    vec[11] = 1.0 if hydroxyl_value not in ('Mono', 'Di', 'Tri') else 0.0  # is_specific
    
    return vec

# Encode all bile acid patterns
bile_acid_encodings = {}
for h in df_activity['Hydroxyl'].unique():
    enc = encode_bile_acid(h)
    bile_acid_encodings[h] = enc
    info = PATTERN_INFO.get(h, (None, None, None, 0, 0))
    print(f"  {h:12s} -> C3={str(info[0]):>4s} C7={str(info[1]):>4s} C12={str(info[2]):>4s}  "
          f"n_OH={info[3]} n_keto={info[4]}  vec={enc.tolist()}")

print(f"\nBile acid positional encoding: {BA_DIM} dims")
print(f"Features: {BA_FEATURE_NAMES}")

## 3. Build Enzyme + Amine + Bile Acid Features

In [ ]:
def get_nonconserved_embedding(enzyme_id, conservation_threshold, pooling='mean'):
    """Extract and pool per-residue embeddings at non-conserved positions."""
    if enzyme_id not in per_residue_embeddings or enzyme_id not in alignment_to_seq:
        return None
    embed = per_residue_embeddings[enzyme_id]
    aln_map = alignment_to_seq[enzyme_id]
    variable_aln_positions = df_core[
        df_core['conservation_score'] < conservation_threshold
    ]['alignment_position'].values
    seq_positions = []
    for aln_pos in variable_aln_positions:
        if aln_pos in aln_map:
            seq_pos = aln_map[aln_pos]
            if seq_pos < len(embed):
                seq_positions.append(seq_pos)
    if len(seq_positions) == 0:
        return None
    selected = embed[seq_positions]
    if pooling == 'mean':
        return selected.mean(axis=0)
    elif pooling == 'max':
        return selected.max(axis=0)
    elif pooling == 'mean_max':
        return np.concatenate([selected.mean(axis=0), selected.max(axis=0)])
    return selected.mean(axis=0)


def get_conserved_embedding(enzyme_id, conservation_threshold=0.95, pooling='mean'):
    """Extract per-residue embeddings at CONSERVED positions (>= threshold)."""
    if enzyme_id not in per_residue_embeddings or enzyme_id not in alignment_to_seq:
        return None
    embed = per_residue_embeddings[enzyme_id]
    aln_map = alignment_to_seq[enzyme_id]
    conserved_aln_positions = df_core[
        df_core['conservation_score'] >= conservation_threshold
    ]['alignment_position'].values
    seq_positions = []
    for aln_pos in conserved_aln_positions:
        if aln_pos in aln_map:
            seq_pos = aln_map[aln_pos]
            if seq_pos < len(embed):
                seq_positions.append(seq_pos)
    if len(seq_positions) == 0:
        return None
    selected = embed[seq_positions]
    if pooling == 'mean':
        return selected.mean(axis=0)
    return selected.mean(axis=0)

print("Embedding functions ready.")

In [ ]:
%%time
# Compute top 4 enzyme representations
enzyme_repr = {}

# 1. full_protein: mean of all residues
enzyme_repr['full_protein'] = {eid: emb for eid, emb in full_embeddings.items()}

# 2. unique: conservation < 0.5, mean pooling
d = {}
for eid in overlap:
    emb = get_nonconserved_embedding(eid, 0.5, pooling='mean')
    if emb is not None:
        d[eid] = emb
enzyme_repr['unique'] = d

# 3. noncons_max: conservation < 0.5, max pooling
d = {}
for eid in overlap:
    emb = get_nonconserved_embedding(eid, 0.5, pooling='max')
    if emb is not None:
        d[eid] = emb
enzyme_repr['noncons_max'] = d

# 4. cons_plus_noncons: conserved(mean) + non-conserved(mean) concatenated
d = {}
for eid in overlap:
    noncons = get_nonconserved_embedding(eid, 0.5, pooling='mean')
    cons = get_conserved_embedding(eid, 0.95, pooling='mean')
    if noncons is not None and cons is not None:
        d[eid] = np.concatenate([cons, noncons])
enzyme_repr['cons_plus_noncons'] = d

print(f"{'Representation':<25s} {'Enzymes':>8s} {'Dims':>6s}")
for name, d in enzyme_repr.items():
    n = len(d)
    dim = list(d.values())[0].shape[0] if n > 0 else 0
    print(f"{name:<25s} {n:>8d} {dim:>6d}")

In [ ]:
# Build amine features: 3 representations
def compute_physicochemical(mol):
    return np.array([
        Descriptors.MolWt(mol),
        Descriptors.MolLogP(mol),
        Descriptors.TPSA(mol),
        Descriptors.NumHDonors(mol),
        Descriptors.NumHAcceptors(mol),
        Descriptors.NumRotatableBonds(mol),
        Descriptors.NumAromaticRings(mol),
        Descriptors.NumAliphaticRings(mol),
        Descriptors.FractionCSP3(mol),
        Descriptors.HeavyAtomCount(mol),
        rdMolDescriptors.CalcNumAmideBonds(mol),
        Descriptors.NumValenceElectrons(mol),
        Descriptors.MaxPartialCharge(mol),
        Descriptors.MinPartialCharge(mol),
        Descriptors.BalabanJ(mol) if Descriptors.BalabanJ(mol) != 0 else 0.0,
    ], dtype=np.float32)

N_PHYSCHEM = 15

# Physicochemical features
repr_physchem = {}
for name, mol in amine_mols.items():
    repr_physchem[name] = compute_physicochemical(mol)
for a in amines_needed:
    if a not in repr_physchem:
        repr_physchem[a] = np.zeros(N_PHYSCHEM, dtype=np.float32)

# One-hot encoding
all_amines_sorted = sorted(amines_needed)
amine_to_idx = {a: i for i, a in enumerate(all_amines_sorted)}
n_amines = len(all_amines_sorted)

repr_onehot = {}
for name in amines_needed:
    vec = np.zeros(n_amines, dtype=np.float32)
    if name in amine_to_idx:
        vec[amine_to_idx[name]] = 1.0
    repr_onehot[name] = vec

# Filter out amines without SMILES
amines_with_smiles = set(amine_mols.keys())

# 1. F_physchem_onehot = physicochemical + one-hot
amine_repr_physchem = {}
for name in amines_needed:
    amine_repr_physchem[name] = np.concatenate([repr_physchem[name], repr_onehot[name]])

# 2. MolT5-small (512-dim)
df_molt5_small = pd.read_csv(DATA_DIR / "molt5_small_amine_embeddings.csv")
molt5_small_dict = {}
for _, row in df_molt5_small.iterrows():
    amine_name = row['amine']
    vec = row.drop('amine').values.astype(np.float32)
    molt5_small_dict[amine_name] = vec

# 3. MolT5-base (768-dim)
df_molt5_base = pd.read_csv(DATA_DIR / "molt5_base_amine_embeddings.csv")
molt5_base_dict = {}
for _, row in df_molt5_base.iterrows():
    amine_name = row['amine']
    vec = row.drop('amine').values.astype(np.float32)
    molt5_base_dict[amine_name] = vec

# Collect all amine representations
amine_repr_all = {
    'physchem_onehot': (amine_repr_physchem, N_PHYSCHEM + n_amines),
    'molt5_small': (molt5_small_dict, 512),
    'molt5_base': (molt5_base_dict, 768),
}

# Determine which amines have embeddings for each representation
print(f"Amine representations:")
for ar_name, (ar_dict, ar_dim) in amine_repr_all.items():
    n_available = sum(1 for a in amines_needed if a in ar_dict and a in amines_with_smiles)
    print(f"  {ar_name:20s}: {ar_dim} dims, {n_available}/{len(amines_needed)} amines available")

In [ ]:
%%time
# Build feature matrices for each enzyme x amine combination
# Each sample = enzyme_embedding + amine_embedding + bile_acid_encoding

feature_matrices = {}
for enz_repr_name, enz_dict in enzyme_repr.items():
    for amine_repr_name, (amine_dict, amine_dim_val) in amine_repr_all.items():
        combo_name = f"{enz_repr_name}__{amine_repr_name}"
        X_list, y_list = [], []
        enzymes, amines, hydroxyls = [], [], []
        for _, row in df_activity.iterrows():
            enzyme, amine, hydroxyl = row['Enzyme'], row['Amine'], row['Hydroxyl']
            if enzyme not in enz_dict:
                continue
            # Skip amines without SMILES or without this amine repr
            if amine not in amines_with_smiles or amine not in amine_dict:
                continue
            enz_emb = enz_dict[enzyme]
            amine_emb = amine_dict[amine]
            ba_emb = bile_acid_encodings[hydroxyl]
            X_list.append(np.concatenate([enz_emb, amine_emb, ba_emb]))
            y_list.append(int(row['active_approach2']))
            enzymes.append(enzyme)
            amines.append(amine)
            hydroxyls.append(hydroxyl)
        
        X = np.array(X_list)
        y = np.array(y_list)
        enz_dim = list(enz_dict.values())[0].shape[0]
        feature_matrices[combo_name] = {
            'X': X, 'y': y,
            'enzymes': enzymes, 'amines': amines, 'hydroxyls': hydroxyls,
            'enz_dim': enz_dim, 'amine_dim': amine_dim_val, 'ba_dim': BA_DIM,
            'total_dims': X.shape[1],
            'enz_repr': enz_repr_name, 'amine_repr': amine_repr_name,
        }

# Print summary
print(f"{'Combo':<45s} {'Samples':>8s} {'Dims':>6s} {'Active':>8s}")
print("-" * 75)
for name, fm in feature_matrices.items():
    print(f"{name:<45s}: {fm['X'].shape[0]:>7d}  {fm['total_dims']:>5d}  {fm['y'].mean():>7.1%}")
print(f"\nTotal combinations: {len(feature_matrices)}")

## 4. Compare: Pair-Level vs Product-Level

In [ ]:
%%time
# Build pair-level features (aggregated, no bile acid) for comparison
# Aggregate: enzyme-amine pair is active if ANY product is active
df_agg = df_activity.groupby(['Enzyme', 'Amine']).agg(
    active=('active_approach2', 'any'),
    n_products=('ProductName', 'count'),
    n_active_products=('active_approach2', 'sum')
).reset_index()

print(f"Pair-level: {len(df_agg)} enzyme-amine pairs, {df_agg['active'].mean():.1%} active")

# Build pair-level feature matrix (using noncons_max + physchem_onehot as representative)
repr_name_comparison = 'noncons_max'
amine_repr_comparison = 'physchem_onehot'
enz_dict = enzyme_repr[repr_name_comparison]
amine_dict, _ = amine_repr_all[amine_repr_comparison]

X_pair, y_pair = [], []
enzymes_pair = []
for _, row in df_agg.iterrows():
    enzyme, amine = row['Enzyme'], row['Amine']
    if enzyme not in enz_dict or amine not in amine_dict or amine not in amines_with_smiles:
        continue
    X_pair.append(np.concatenate([enz_dict[enzyme], amine_dict[amine]]))
    y_pair.append(int(row['active']))
    enzymes_pair.append(enzyme)

X_pair = np.array(X_pair)
y_pair = np.array(y_pair)
print(f"Pair-level feature matrix: {X_pair.shape}, active={y_pair.mean():.1%}")

# Product-level features (with bile acid)
fm = feature_matrices[f'{repr_name_comparison}__{amine_repr_comparison}']
print(f"Product-level feature matrix: {fm['X'].shape}, active={fm['y'].mean():.1%}")

In [ ]:
%%time
def enzyme_holdout_split_seed(X, y, enzymes, seed, test_size=0.2, val_size=0.2):
    """Enzyme hold-out split with a specific random seed."""
    enzymes_arr = np.array(enzymes)
    unique_enzymes = np.unique(enzymes_arr)
    profiles = np.array([y[enzymes_arr == e].mean() for e in unique_enzymes])
    bins = pd.cut(profiles, bins=5, labels=False)
    
    # Fall back to non-stratified if too few enzymes per bin
    min_bin_count = pd.Series(bins).value_counts().min() if len(bins) > 1 else 0
    stratify_1 = bins if min_bin_count >= 2 else None
    
    train_val_enz, test_enz = train_test_split(
        unique_enzymes, test_size=test_size, random_state=seed, stratify=stratify_1
    )
    tv_profiles = np.array([y[enzymes_arr == e].mean() for e in train_val_enz])
    tv_bins = pd.cut(tv_profiles, bins=5, labels=False)
    
    min_tv_count = pd.Series(tv_bins).value_counts().min() if len(tv_bins) > 1 else 0
    stratify_2 = tv_bins if min_tv_count >= 2 else None
    
    train_enz, val_enz = train_test_split(
        train_val_enz, test_size=val_size, random_state=seed, stratify=stratify_2
    )
    train_mask = np.isin(enzymes_arr, train_enz)
    val_mask = np.isin(enzymes_arr, val_enz)
    test_mask = np.isin(enzymes_arr, test_enz)
    return {
        'X_train': X[train_mask], 'y_train': y[train_mask],
        'X_val': X[val_mask], 'y_val': y[val_mask],
        'X_test': X[test_mask], 'y_test': y[test_mask],
        'train_enz': train_enz, 'val_enz': val_enz, 'test_enz': test_enz,
    }

# Compare pair-level vs product-level over 3 seeds
comparison_results = []
for seed in SEEDS[:3]:
    # Pair-level
    split_pair = enzyme_holdout_split_seed(X_pair, y_pair, enzymes_pair, seed)
    model_pair = xgb.XGBClassifier(
        n_estimators=300, max_depth=3, learning_rate=0.05,
        scale_pos_weight=(split_pair['y_train'] == 0).sum() / max((split_pair['y_train'] == 1).sum(), 1),
        reg_alpha=1.0, reg_lambda=5.0, subsample=0.7, colsample_bytree=0.7,
        min_child_weight=5, random_state=42, early_stopping_rounds=30,
        eval_metric='logloss', n_jobs=-1
    )
    model_pair.fit(split_pair['X_train'], split_pair['y_train'],
                   eval_set=[(split_pair['X_val'], split_pair['y_val'])], verbose=False)
    y_proba_pair = model_pair.predict_proba(split_pair['X_test'])[:, 1]
    y_pred_pair = model_pair.predict(split_pair['X_test'])
    
    # Product-level
    split_prod = enzyme_holdout_split_seed(fm['X'], fm['y'], fm['enzymes'], seed)
    model_prod = xgb.XGBClassifier(
        n_estimators=300, max_depth=3, learning_rate=0.05,
        scale_pos_weight=(split_prod['y_train'] == 0).sum() / max((split_prod['y_train'] == 1).sum(), 1),
        reg_alpha=1.0, reg_lambda=5.0, subsample=0.7, colsample_bytree=0.7,
        min_child_weight=5, random_state=42, early_stopping_rounds=30,
        eval_metric='logloss', n_jobs=-1
    )
    model_prod.fit(split_prod['X_train'], split_prod['y_train'],
                   eval_set=[(split_prod['X_val'], split_prod['y_val'])], verbose=False)
    y_proba_prod = model_prod.predict_proba(split_prod['X_test'])[:, 1]
    y_pred_prod = model_prod.predict(split_prod['X_test'])
    
    comparison_results.append({
        'seed': seed,
        'pair_roc_auc': roc_auc_score(split_pair['y_test'], y_proba_pair),
        'pair_pr_auc': average_precision_score(split_pair['y_test'], y_proba_pair),
        'pair_f1': f1_score(split_pair['y_test'], y_pred_pair, zero_division=0),
        'pair_samples': len(split_pair['y_test']),
        'prod_roc_auc': roc_auc_score(split_prod['y_test'], y_proba_prod),
        'prod_pr_auc': average_precision_score(split_prod['y_test'], y_proba_prod),
        'prod_f1': f1_score(split_prod['y_test'], y_pred_prod, zero_division=0),
        'prod_samples': len(split_prod['y_test']),
    })

df_comp = pd.DataFrame(comparison_results)
print("\nPair-level vs Product-level (noncons_max + physchem_onehot, 3 seeds):")
print(f"  Pair-level    — ROC-AUC: {df_comp['pair_roc_auc'].mean():.3f} +/- {df_comp['pair_roc_auc'].std():.3f}, "
      f"PR-AUC: {df_comp['pair_pr_auc'].mean():.3f} +/- {df_comp['pair_pr_auc'].std():.3f}")
print(f"  Product-level — ROC-AUC: {df_comp['prod_roc_auc'].mean():.3f} +/- {df_comp['prod_roc_auc'].std():.3f}, "
      f"PR-AUC: {df_comp['prod_pr_auc'].mean():.3f} +/- {df_comp['prod_pr_auc'].std():.3f}")

In [ ]:
# Plot pair vs product comparison
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
metrics_comp = ['roc_auc', 'pr_auc', 'f1']
titles = ['ROC-AUC', 'PR-AUC', 'F1 Score']

for ax, metric, title in zip(axes, metrics_comp, titles):
    pair_vals = df_comp[f'pair_{metric}'].values
    prod_vals = df_comp[f'prod_{metric}'].values
    x = np.arange(len(pair_vals))
    width = 0.35
    ax.bar(x - width/2, pair_vals, width, label='Pair-level', color='steelblue', alpha=0.8)
    ax.bar(x + width/2, prod_vals, width, label='Product-level', color='darkorange', alpha=0.8)
    ax.set_xlabel('Seed')
    ax.set_ylabel(title)
    ax.set_title(title)
    ax.set_xticks(x)
    ax.set_xticklabels([str(s) for s in df_comp['seed']])
    ax.legend()
    # Add mean lines
    ax.axhline(pair_vals.mean(), color='steelblue', linestyle='--', alpha=0.5)
    ax.axhline(prod_vals.mean(), color='darkorange', linestyle='--', alpha=0.5)

plt.suptitle(f'Pair-Level vs Product-Level ({repr_name_comparison})', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(BA_DIR / 'pair_vs_product_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: pair_vs_product_comparison.png")

## 5. Train XGBoost + Random Forest + MLP (Product-Level)

In [ ]:
def train_xgb_with_logloss(split_data):
    """Train regularized XGBoost, tracking train+val log loss."""
    X_train, y_train = split_data['X_train'], split_data['y_train']
    X_val, y_val = split_data['X_val'], split_data['y_val']
    X_test, y_test = split_data['X_test'], split_data['y_test']
    
    n_neg = (y_train == 0).sum()
    n_pos = max((y_train == 1).sum(), 1)
    
    model = xgb.XGBClassifier(
        n_estimators=300, max_depth=3, learning_rate=0.05,
        scale_pos_weight=n_neg / n_pos,
        reg_alpha=1.0, reg_lambda=5.0,
        subsample=0.7, colsample_bytree=0.7,
        min_child_weight=5,
        random_state=42, early_stopping_rounds=30,
        eval_metric='logloss', n_jobs=-1
    )
    model.fit(
        X_train, y_train,
        eval_set=[(X_train, y_train), (X_val, y_val)],
        verbose=False
    )
    
    evals = model.evals_result()
    train_logloss = evals['validation_0']['logloss']
    val_logloss = evals['validation_1']['logloss']
    
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    y_proba_train = model.predict_proba(X_train)[:, 1]
    y_proba_val = model.predict_proba(X_val)[:, 1]
    
    metrics = {
        'train_logloss': log_loss(y_train, y_proba_train),
        'val_logloss': log_loss(y_val, y_proba_val),
        'test_logloss': log_loss(y_test, y_proba),
        'logloss_gap': log_loss(y_val, y_proba_val) - log_loss(y_train, y_proba_train),
        'roc_auc': roc_auc_score(y_test, y_proba),
        'pr_auc': average_precision_score(y_test, y_proba),
        'f1': f1_score(y_test, y_pred, zero_division=0),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'accuracy': accuracy_score(y_test, y_pred),
        'train_acc': model.score(X_train, y_train),
        'val_acc': model.score(X_val, y_val),
        'n_rounds': len(train_logloss),
    }
    return metrics, train_logloss, val_logloss, model


def train_rf(split_data):
    """Train Random Forest classifier."""
    X_train, y_train = split_data['X_train'], split_data['y_train']
    X_test, y_test = split_data['X_test'], split_data['y_test']
    
    model = RandomForestClassifier(
        n_estimators=300, max_depth=10, min_samples_leaf=5,
        class_weight='balanced', random_state=42, n_jobs=-1
    )
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    metrics = {
        'roc_auc': roc_auc_score(y_test, y_proba),
        'pr_auc': average_precision_score(y_test, y_proba),
        'f1': f1_score(y_test, y_pred, zero_division=0),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'accuracy': accuracy_score(y_test, y_pred),
    }
    return metrics, model


def train_mlp(split_data):
    """Train MLP classifier with feature standardization."""
    X_train, y_train = split_data['X_train'], split_data['y_train']
    X_val, y_val = split_data['X_val'], split_data['y_val']
    X_test, y_test = split_data['X_test'], split_data['y_test']
    
    # Standardize features (critical for MLP)
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_val_s = scaler.transform(X_val)
    X_test_s = scaler.transform(X_test)
    
    # Combine train + val for fitting (sklearn handles its own internal early stopping split)
    # Test set remains untouched for evaluation
    X_fit = np.vstack([X_train_s, X_val_s])
    y_fit = np.concatenate([y_train, y_val])
    
    model = MLPClassifier(
        hidden_layer_sizes=(256, 128, 64),
        activation='relu',
        alpha=0.001,          # L2 regularization
        batch_size=256,
        learning_rate='adaptive',
        learning_rate_init=0.001,
        max_iter=300,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=20,
        random_state=42,
    )
    model.fit(X_fit, y_fit)
    
    y_pred = model.predict(X_test_s)
    y_proba = model.predict_proba(X_test_s)[:, 1]
    
    # Also evaluate on train for overfitting check
    y_proba_train = model.predict_proba(X_train_s)[:, 1]
    y_proba_val = model.predict_proba(X_val_s)[:, 1]
    
    metrics = {
        'roc_auc': roc_auc_score(y_test, y_proba),
        'pr_auc': average_precision_score(y_test, y_proba),
        'f1': f1_score(y_test, y_pred, zero_division=0),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'accuracy': accuracy_score(y_test, y_pred),
        'train_logloss': log_loss(y_train, y_proba_train),
        'val_logloss': log_loss(y_val, y_proba_val),
        'test_logloss': log_loss(y_test, y_proba),
        'logloss_gap': log_loss(y_val, y_proba_val) - log_loss(y_train, y_proba_train),
        'n_iter': model.n_iter_,
    }
    return metrics, model, scaler

print("Training functions ready (XGBoost, RF, MLP).")

In [ ]:
%%time
# Train all: 4 enzyme repr x 3 amine repr x 10 splits x 3 models = 360 experiments
all_results = []
all_logloss_curves = {}
all_models = {}

for combo_name, fm in feature_matrices.items():
    X, y = fm['X'], fm['y']
    enzymes_list = fm['enzymes']
    print(f"\n--- {combo_name} ({fm['total_dims']} dims, {fm['X'].shape[0]} samples) ---")
    
    for i, seed in enumerate(SEEDS):
        split = enzyme_holdout_split_seed(X, y, enzymes_list, seed)
        
        # XGBoost
        xgb_metrics, train_ll, val_ll, xgb_model = train_xgb_with_logloss(split)
        xgb_metrics['combo'] = combo_name
        xgb_metrics['enz_repr'] = fm['enz_repr']
        xgb_metrics['amine_repr'] = fm['amine_repr']
        xgb_metrics['model'] = 'XGBoost'
        xgb_metrics['split_idx'] = i
        xgb_metrics['seed'] = seed
        all_results.append(xgb_metrics)
        all_logloss_curves[(combo_name, i)] = (train_ll, val_ll)
        all_models[(combo_name, 'xgb', i)] = xgb_model
        
        # Random Forest
        rf_metrics, rf_model = train_rf(split)
        rf_metrics['combo'] = combo_name
        rf_metrics['enz_repr'] = fm['enz_repr']
        rf_metrics['amine_repr'] = fm['amine_repr']
        rf_metrics['model'] = 'RandomForest'
        rf_metrics['split_idx'] = i
        rf_metrics['seed'] = seed
        all_results.append(rf_metrics)
        all_models[(combo_name, 'rf', i)] = rf_model
        
        # MLP
        mlp_metrics, mlp_model, mlp_scaler = train_mlp(split)
        mlp_metrics['combo'] = combo_name
        mlp_metrics['enz_repr'] = fm['enz_repr']
        mlp_metrics['amine_repr'] = fm['amine_repr']
        mlp_metrics['model'] = 'MLP'
        mlp_metrics['split_idx'] = i
        mlp_metrics['seed'] = seed
        all_results.append(mlp_metrics)
        all_models[(combo_name, 'mlp', i)] = (mlp_model, mlp_scaler)
    
    # Print summary for this combo
    combo_xgb = [r for r in all_results if r['combo'] == combo_name and r['model'] == 'XGBoost']
    combo_rf = [r for r in all_results if r['combo'] == combo_name and r['model'] == 'RandomForest']
    combo_mlp = [r for r in all_results if r['combo'] == combo_name and r['model'] == 'MLP']
    print(f"  XGBoost  — ROC-AUC: {np.mean([r['roc_auc'] for r in combo_xgb]):.3f}, "
          f"PR-AUC: {np.mean([r['pr_auc'] for r in combo_xgb]):.3f}, "
          f"Gap: {np.mean([r['logloss_gap'] for r in combo_xgb]):+.3f}")
    print(f"  RF       — ROC-AUC: {np.mean([r['roc_auc'] for r in combo_rf]):.3f}, "
          f"PR-AUC: {np.mean([r['pr_auc'] for r in combo_rf]):.3f}")
    print(f"  MLP      — ROC-AUC: {np.mean([r['roc_auc'] for r in combo_mlp]):.3f}, "
          f"PR-AUC: {np.mean([r['pr_auc'] for r in combo_mlp]):.3f}, "
          f"Gap: {np.mean([r['logloss_gap'] for r in combo_mlp]):+.3f}")

df_results = pd.DataFrame(all_results)
print(f"\nTotal experiments: {len(df_results)}")

## 6. Feature Importance (Enzyme vs Amine vs Bile Acid)

In [ ]:
# Compute block-level feature importance from XGBoost models
block_importance_data = []

for combo_name, fm in feature_matrices.items():
    enz_dim = fm['enz_dim']
    amine_dim_val = fm['amine_dim']
    ba_dim = fm['ba_dim']
    
    for i in range(len(SEEDS)):
        model = all_models.get((combo_name, 'xgb', i))
        if model is None:
            continue
        
        importances = model.feature_importances_
        
        enz_imp = importances[:enz_dim].sum()
        amine_imp = importances[enz_dim:enz_dim + amine_dim_val].sum()
        ba_imp = importances[enz_dim + amine_dim_val:].sum()
        total_imp = enz_imp + amine_imp + ba_imp
        
        if total_imp > 0:
            block_importance_data.append({
                'combo': combo_name,
                'enz_repr': fm['enz_repr'],
                'amine_repr': fm['amine_repr'],
                'split_idx': i,
                'enzyme_pct': 100 * enz_imp / total_imp,
                'amine_pct': 100 * amine_imp / total_imp,
                'bile_acid_pct': 100 * ba_imp / total_imp,
            })

df_block_imp = pd.DataFrame(block_importance_data)

# Summary by amine representation (averaged across enzyme repr and splits)
print("Block-Level Feature Importance by Amine Representation:")
print("=" * 70)
for ar_name in amine_repr_all.keys():
    subset = df_block_imp[df_block_imp['amine_repr'] == ar_name]
    print(f"\n{ar_name} (avg across enzyme repr):")
    print(f"  Enzyme:    {subset['enzyme_pct'].mean():5.1f}% +/- {subset['enzyme_pct'].std():4.1f}%")
    print(f"  Amine:     {subset['amine_pct'].mean():5.1f}% +/- {subset['amine_pct'].std():4.1f}%")
    print(f"  Bile acid: {subset['bile_acid_pct'].mean():5.1f}% +/- {subset['bile_acid_pct'].std():4.1f}%")

In [ ]:
# Stacked bar chart: block importance per combo (grouped by amine repr)
combo_names = list(feature_matrices.keys())
n_combos = len(combo_names)

fig, ax = plt.subplots(figsize=(max(14, n_combos * 1.2), 6))
x = np.arange(n_combos)
width = 0.6

enz_means = [df_block_imp[df_block_imp['combo'] == c]['enzyme_pct'].mean() for c in combo_names]
amine_means = [df_block_imp[df_block_imp['combo'] == c]['amine_pct'].mean() for c in combo_names]
ba_means = [df_block_imp[df_block_imp['combo'] == c]['bile_acid_pct'].mean() for c in combo_names]

ax.bar(x, enz_means, width, label='Enzyme', color='steelblue')
ax.bar(x, amine_means, width, bottom=enz_means, label='Amine', color='darkorange')
ax.bar(x, ba_means, width, bottom=[e + a for e, a in zip(enz_means, amine_means)], 
       label='Bile acid', color='forestgreen')

# Short labels: enz_repr + amine_repr abbreviation
short_labels = []
for c in combo_names:
    enz, amine = c.split('__')
    amine_short = {'physchem_onehot': 'phys', 'molt5_small': 'm5s', 'molt5_base': 'm5b'}[amine]
    short_labels.append(f"{enz}\n({amine_short})")

ax.set_xlabel('Enzyme + Amine Representation')
ax.set_ylabel('Feature Importance (%)')
ax.set_title('Feature Block Importance (XGBoost, Product-Level)')
ax.set_xticks(x)
ax.set_xticklabels(short_labels, rotation=45, ha='right', fontsize=7)
ax.legend()
ax.set_ylim(0, 105)

# Add amine % labels
for i, (e, a, b) in enumerate(zip(enz_means, amine_means, ba_means)):
    if a > 3:
        ax.text(i, e + a/2, f'{a:.0f}%', ha='center', va='center', fontsize=6, color='white', fontweight='bold')
    if b > 1.5:
        ax.text(i, e + a + b/2, f'{b:.1f}%', ha='center', va='center', fontsize=6, color='white', fontweight='bold')

plt.tight_layout()
plt.savefig(BA_DIR / 'feature_importance_3blocks.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: feature_importance_3blocks.png")

In [ ]:
# Bile acid feature detail: which positional/count features are most important?
ba_detail_data = []

for combo_name, fm in feature_matrices.items():
    enz_dim = fm['enz_dim']
    amine_dim_val = fm['amine_dim']
    ba_start = enz_dim + amine_dim_val
    
    for i in range(len(SEEDS)):
        model = all_models.get((combo_name, 'xgb', i))
        if model is None:
            continue
        importances = model.feature_importances_
        ba_importances = importances[ba_start:ba_start + BA_DIM]
        ba_total = ba_importances.sum()
        if ba_total > 0:
            for j, fname in enumerate(BA_FEATURE_NAMES):
                ba_detail_data.append({
                    'combo': combo_name,
                    'split_idx': i,
                    'feature': fname,
                    'importance': ba_importances[j],
                    'pct_of_ba_block': 100 * ba_importances[j] / ba_total,
                })

df_ba_detail = pd.DataFrame(ba_detail_data)

# Aggregate across splits and combos
ba_avg = df_ba_detail.groupby('feature')['pct_of_ba_block'].mean().sort_values(ascending=False)
print("Bile Acid Feature Importance (% within BA block, avg across all combos):")
for feat, pct in ba_avg.items():
    bar = '#' * int(pct / 2)
    print(f"  {feat:12s}: {pct:5.1f}% {bar}")

# Plot — group by feature type
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: positional features (C3, C7, C12)
pos_features = [f for f in BA_FEATURE_NAMES[:9]]
pos_vals = [ba_avg.get(f, 0) for f in pos_features]
colors_pos = ['#2196F3', '#FF5722', '#9E9E9E'] * 3  # aOH=blue, keto=red, unspec=gray
axes[0].bar(range(len(pos_features)), pos_vals, color=colors_pos, edgecolor='black', alpha=0.8)
axes[0].set_xticks(range(len(pos_features)))
axes[0].set_xticklabels(pos_features, rotation=45, ha='right', fontsize=8)
axes[0].set_ylabel('Importance (% within BA block)')
axes[0].set_title('Positional Features (C3/C7/C12)\nBlue=alpha-OH, Red=keto, Gray=unspecified')

# Right: count features
count_features = BA_FEATURE_NAMES[9:]
count_vals = [ba_avg.get(f, 0) for f in count_features]
axes[1].bar(range(len(count_features)), count_vals, color=['#4CAF50', '#FF9800', '#673AB7'], 
            edgecolor='black', alpha=0.8)
axes[1].set_xticks(range(len(count_features)))
axes[1].set_xticklabels(count_features, fontsize=10)
axes[1].set_ylabel('Importance (% within BA block)')
axes[1].set_title('Count Features')

plt.suptitle('Bile Acid Feature Importance (Positional Encoding)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(BA_DIR / 'bile_acid_pattern_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: bile_acid_pattern_importance.png")

## 7. Performance Comparison

In [ ]:
# Performance comparison: grouped by amine representation (3 models)
amine_reprs = list(amine_repr_all.keys())
enz_reprs = list(enzyme_repr.keys())
metrics_to_plot = ['roc_auc', 'pr_auc', 'f1']
titles = ['ROC-AUC', 'PR-AUC', 'F1 Score']
model_names = ['XGBoost', 'RandomForest', 'MLP']
model_colors = ['steelblue', 'darkorange', '#2ca02c']

fig, axes = plt.subplots(len(amine_reprs), 3, figsize=(18, 5 * len(amine_reprs)))

for row_idx, ar_name in enumerate(amine_reprs):
    for col_idx, (metric, title) in enumerate(zip(metrics_to_plot, titles)):
        ax = axes[row_idx, col_idx]
        x = np.arange(len(enz_reprs))
        n_models = len(model_names)
        width = 0.25
        
        for m_idx, (m_name, m_color) in enumerate(zip(model_names, model_colors)):
            means, stds = [], []
            for er_name in enz_reprs:
                combo = f"{er_name}__{ar_name}"
                vals = df_results[(df_results['combo'] == combo) & (df_results['model'] == m_name)][metric]
                means.append(vals.mean())
                stds.append(vals.std())
            offset = (m_idx - 1) * width
            ax.bar(x + offset, means, width, yerr=stds, label=m_name,
                   color=m_color, alpha=0.8, capsize=2)
        
        ax.set_xticks(x)
        ax.set_xticklabels(enz_reprs, rotation=30, ha='right', fontsize=8)
        ax.set_ylabel(title)
        if row_idx == 0:
            ax.set_title(title)
        if col_idx == 0:
            ax.set_ylabel(f"{ar_name}\n{title}")
        ax.legend(fontsize=6)

plt.suptitle('Product-Level Model Performance by Amine Representation', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(BA_DIR / 'product_level_comparison_bars.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: product_level_comparison_bars.png")

In [ ]:
# Box plots: PR-AUC by amine representation (3 models)
fig, axes = plt.subplots(1, len(amine_reprs), figsize=(7 * len(amine_reprs), 5))

model_box_colors = ['steelblue', 'darkorange', '#2ca02c']

for ax, ar_name in zip(axes, amine_reprs):
    all_data = {m: [] for m in model_names}
    labels = []
    for er_name in enz_reprs:
        combo = f"{er_name}__{ar_name}"
        labels.append(er_name)
        for m_name in model_names:
            all_data[m_name].append(
                df_results[(df_results['combo'] == combo) & (df_results['model'] == m_name)]['pr_auc'].values
            )
    
    n_enz = len(enz_reprs)
    group_width = len(model_names) * 0.6 + 0.5
    bp_handles = []
    for m_idx, (m_name, m_color) in enumerate(zip(model_names, model_box_colors)):
        positions = np.arange(n_enz) * group_width + m_idx * 0.6
        bp = ax.boxplot(all_data[m_name], positions=positions, widths=0.45, patch_artist=True)
        for patch in bp['boxes']:
            patch.set_facecolor(m_color)
            patch.set_alpha(0.7)
        bp_handles.append(bp['boxes'][0])
    
    center_positions = np.arange(n_enz) * group_width + 0.6
    ax.set_xticks(center_positions)
    ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=8)
    ax.set_ylabel('PR-AUC')
    ax.set_title(f'Amine: {ar_name}')
    ax.legend(bp_handles, model_names, fontsize=7)

plt.suptitle('PR-AUC Distribution (Product-Level, 10 Splits)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(BA_DIR / 'product_level_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: product_level_boxplots.png")

In [ ]:
# Log loss convergence curves (XGBoost only, grouped by amine repr)
fig, axes = plt.subplots(len(amine_reprs), len(enz_reprs), 
                         figsize=(4 * len(enz_reprs), 3.5 * len(amine_reprs)))

for row_idx, ar_name in enumerate(amine_reprs):
    for col_idx, er_name in enumerate(enz_reprs):
        ax = axes[row_idx, col_idx]
        combo = f"{er_name}__{ar_name}"
        for i in range(len(SEEDS)):
            if (combo, i) not in all_logloss_curves:
                continue
            train_ll, val_ll = all_logloss_curves[(combo, i)]
            ax.plot(train_ll, color='steelblue', alpha=0.3, linewidth=0.8)
            ax.plot(val_ll, color='darkorange', alpha=0.3, linewidth=0.8)
        
        ax.plot([], color='steelblue', label='Train')
        ax.plot([], color='darkorange', label='Val')
        ax.set_xlabel('Round', fontsize=7)
        ax.set_ylabel('Log Loss', fontsize=7)
        ax.set_title(f"{er_name}\n({ar_name})", fontsize=7)
        ax.legend(fontsize=6)
        ax.tick_params(labelsize=6)

plt.suptitle('XGBoost Log Loss Convergence (Product-Level)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(BA_DIR / 'logloss_convergence_product.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: logloss_convergence_product.png")

In [ ]:
# Summary table
summary_rows = []
for combo_name, fm in feature_matrices.items():
    for model_name in ['XGBoost', 'RandomForest', 'MLP']:
        subset = df_results[(df_results['combo'] == combo_name) & (df_results['model'] == model_name)]
        if len(subset) == 0:
            continue
        row = {
            'combo': combo_name,
            'enz_repr': fm['enz_repr'],
            'amine_repr': fm['amine_repr'],
            'model': model_name,
            'total_dims': fm['total_dims'],
            'enz_dims': fm['enz_dim'],
            'amine_dims': fm['amine_dim'],
            'roc_auc_mean': subset['roc_auc'].mean(),
            'roc_auc_std': subset['roc_auc'].std(),
            'pr_auc_mean': subset['pr_auc'].mean(),
            'pr_auc_std': subset['pr_auc'].std(),
            'f1_mean': subset['f1'].mean(),
            'f1_std': subset['f1'].std(),
            'accuracy_mean': subset['accuracy'].mean(),
        }
        if 'logloss_gap' in subset.columns and subset['logloss_gap'].notna().any():
            row['logloss_gap_mean'] = subset['logloss_gap'].mean()
            row['logloss_gap_std'] = subset['logloss_gap'].std()
        if 'val_logloss' in subset.columns and subset['val_logloss'].notna().any():
            row['val_logloss_mean'] = subset['val_logloss'].mean()
        if 'train_logloss' in subset.columns and subset['train_logloss'].notna().any():
            row['train_logloss_mean'] = subset['train_logloss'].mean()
        summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows)

# Print ranked by PR-AUC
print("\n" + "=" * 105)
print("PRODUCT-LEVEL MODEL SUMMARY (ranked by PR-AUC)")
print("=" * 105)
df_ranked = df_summary.sort_values('pr_auc_mean', ascending=False)
for _, row in df_ranked.head(25).iterrows():
    gap_str = ''
    if 'logloss_gap_mean' in row and pd.notna(row.get('logloss_gap_mean')):
        gap_str = f", Gap: {row['logloss_gap_mean']:+.3f}"
    print(f"{row['combo']:40s} {row['model']:12s} — ROC: {row['roc_auc_mean']:.3f}, "
          f"PR: {row['pr_auc_mean']:.3f}+/-{row['pr_auc_std']:.3f}, "
          f"F1: {row['f1_mean']:.3f}{gap_str}")

## 8. Save Results

In [ ]:
# Save all results
df_results.to_csv(BA_DIR / 'bile_acid_hydroxylation_results.csv', index=False)
df_summary.to_csv(BA_DIR / 'bile_acid_hydroxylation_summary.csv', index=False)
df_block_imp.to_csv(BA_DIR / 'feature_importance_blocks.csv', index=False)
df_ba_detail.to_csv(BA_DIR / 'bile_acid_feature_detail.csv', index=False)
df_comp.to_csv(BA_DIR / 'pair_vs_product_comparison.csv', index=False)

print(f"Saved results to {BA_DIR}/")
print(f"  bile_acid_hydroxylation_results.csv — {len(df_results)} experiment rows")
print(f"  bile_acid_hydroxylation_summary.csv — {len(df_summary)} summary rows")
print(f"  feature_importance_blocks.csv — block-level importance")
print(f"  bile_acid_feature_detail.csv — per-feature BA detail")
print(f"  pair_vs_product_comparison.csv — pair vs product comparison")

# Best models
print("\n--- Top 5 by PR-AUC ---")
for _, row in df_ranked.head(5).iterrows():
    print(f"  {row['combo']:40s} {row['model']:12s} — PR-AUC: {row['pr_auc_mean']:.3f}, ROC-AUC: {row['roc_auc_mean']:.3f}")